In [1]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
paths = !ls pile-tokenized/data

In [3]:
steps = 1000
batch = 1024
seq = 2049
macro_steps = 95

MASSIVE_TOKENS = np.memmap('pile-tokenized/massive_tokens.npy', dtype=np.uint16, mode='w+', shape=(macro_steps, steps * batch, seq))

In [4]:
LOCK = Lock()

def process_macro_step(i):
  path = f'pile-tokenized/data/train-{i * 1000:0>6}.parquet'
  df = pd.read_parquet(path)
  tokens = np.stack(df.token_ids.values)
  with LOCK:
    MASSIVE_TOKENS[i - 1] = tokens


with ThreadPoolExecutor(6) as ex:
  list(tqdm(ex.map(process_macro_step, range(1, 96))))


0it [00:00, ?it/s]

In [18]:
MASSIVE_TOKENS.shape

(95, 1024000, 2049)